In [6]:
import os, sys, json

sys.path.append("../")
sys.path.append("../src/")
sys.path.append("../data/")
sys.path.append("../model_evaluation")
sys.path.append("../model_evaluation/evaluation")

In [7]:
from text_evaluation import text_similarity as ts
import bpmn_similarity

In [8]:
path_text_1 = "../data/edge_case/original/generated_process_description/"
path_text_2 = "../data/edge_case/changed/generated_process_description/"
# path_text_1 = "../data/edge_case/original/process_descriptions/"
# path_text_2 = "../data/edge_case/changed/process_descriptions/"
path_model_1 = "../data/edge_case/original/ground_json/"
path_model_2 = "../data/edge_case/changed/ground_json/"

In [9]:
eval = {"text_sts": [], "text_sentence_sim": [], "model": []}
for i in ["1_2", "3_3", "10_6"]:
    with open(path_text_1 + i + ".txt", "r") as file:
        file_1 = file.read()

    with open(path_text_2 + i + ".txt", "r") as file:
        file_2 = file.read()

    with open(path_model_1 + i + ".json", "r") as infile:
        g1 = json.load(infile)

    with open(path_model_2 + i + ".json", "r") as infile:
        g2 = json.load(infile)

    sentences_1 = ts.get_sentences(file_1)
    sentences_2 = ts.get_sentences(file_2)

    adjusted_sen2 = ts.align_sentences(sentences_1, sentences_2, threshold=0.8)
    sequence_similarity = ts.sequence_similarity(sentences_1, adjusted_sen2)
    overalll_sim = 0.5 * ts.sts_bert(file_1, file_2) + 0.5 * sequence_similarity

    eval["text_sts"].append(ts.sts_bert(file_1, file_2))
    eval["text_sentence_sim"].append(overalll_sim)
    eval["model"].append(
        bpmn_similarity.calculate_similarity_scores(g1, g2, method="dice", similarity_threshold=0.99)[0]["overall"]
    )

In [10]:
eval

{'text_sts': [0.95, 0.88, 0.89],
 'text_sentence_sim': [0.725, 0.6672727272727272, 0.5813636363636363],
 'model': [0.94, 0.8709677419354839, 0.8636363636363636]}